# Fase 3 — Núcleo algorítmico: TMDB 2010–2025
**Grupo 4 · MCDI500 · Notebook de integración**

Refuerza trazabilidad, pruebas, recursividad, eficiencia y POO. El objetivo es preparar un pipeline reutilizable como base de la Fase 4; no se afirma que el recomendador final esté implementado.

**Mejoras:** auditoría de datos; partición reproducible; transformadores con interfaz común; parámetros aprendidos solo en entrenamiento; Weighted Rating; vectorización dispersa; recursividad e iteración; benchmarks; pruebas y persistencia.

Ejecutar las celdas en orden. Se conserva `df_raw` y las transformaciones trabajan sobre copias. El CSV debe estar en `data/raw/movies_raw.csv` o se debe modificar `RUTA_DATOS`.

## 0. Configuración y carga de dependencias

Se concentraron todas las decisiones numéricas en constantes al inicio: `SEMILLA`, `FRACCION_PRUEBA`, `PERCENTIL_M`, `RUNTIME_MIN/MAX`, `MIN_GRUPO_RUNTIME`, `UMBRAL_DINERO` y `MAX_PRODUCTORAS`. También se hizo que la ruta del CSV se encuentre sola

**Por qué.**
- *Reproducibilidad.* Con `SEMILLA = 42` la partición train/test es siempre la misma; sin ella, cada ejecución cambiaría los parámetros aprendidos (C, m, medianas) y no podría comparar resultados entre corridas ni entre integrantes del grupo.
- *Trazabilidad.* Un número escrito en medio de una función (un `0.75`, un `1000`) es una decisión escondida. Al nombrarlo, cualquiera puede ver qué supuesto hice y cambiarlo en un solo lugar. Estas constantes además se guardan en el JSON final (sección 12).



In [1]:
from pathlib import Path
import json, time, tracemalloc, platform, statistics, sys
from abc import ABC, abstractmethod
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import display

SEMILLA = 42
FRACCION_PRUEBA = 0.20
PERCENTIL_M = 0.75
RUNTIME_MIN, RUNTIME_MAX = 40, 300   # rango plausible de duración (min) para APRENDER medianas
MIN_GRUPO_RUNTIME = 30               # mínimo de filas válidas para usar la mediana de un género
UMBRAL_DINERO = 1000                 # budget/revenue por debajo de esto se tratan como ausentes
MAX_PRODUCTORAS = 3
COLUMNAS = ["movie_id","title","release_date","status","budget","revenue","runtime",
            "genres","production_companies","production_countries","spoken_languages",
            "original_language","vote_average","vote_count","popularity","adult"]
NUMERICAS = ["movie_id","budget","revenue","runtime","vote_average","vote_count","popularity"]

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatas if (p/"data").is_dir()), Path.cwd())
_nombres = ["movies_raw.csv"]
_cands = [RAIZ/"data"/"raw"/n for n in _nombres] + [Path.cwd()/n for n in _nombres]
RUTA_DATOS = next((p for p in _cands if p.is_file()), RAIZ/"data"/"raw"/"movies_raw.csv")
DIR_SALIDA = RAIZ/"F3"/"results_mejorado"
print("Raíz:", RAIZ, "| Python:", platform.python_version(), "| intérprete:", sys.executable)
print("CSV:", RUTA_DATOS)

Raíz: /Users/jocelynmanriquez/Desktop/MagisterDatosIA/proyectos/mcdi500-tmdb-grupo4 | Python: 3.12.4 | intérprete: /Users/jocelynmanriquez/Desktop/MagisterDatosIA/proyectos/mcdi500-tmdb-grupo4/.venv/bin/python
CSV: /Users/jocelynmanriquez/Desktop/MagisterDatosIA/proyectos/mcdi500-tmdb-grupo4/data/raw/movies_raw.csv


## 1. Carga y auditoría inicial
Se validan archivo, esquema e identificadores. La conversión numérica es explícita; los valores no convertibles quedan como nulos y se contabilizan.

### ¿Qué se hizo?

Antes de realizar cualquier transformación, se llevaron a cabo las siguientes comprobaciones:

1. Se verificó que el archivo existiera en la ruta indicada.
2. Se comprobó que estuvieran presentes las 16 columnas esperadas.
3. Se convirtieron las columnas numéricas utilizando `errors="coerce"`, de modo que los valores no convertibles quedaran registrados como nulos.
4. Se validó que `movie_id` fuera numérico, no contuviera valores nulos y que todos sus registros fueran únicos.

Posteriormente, se revisaron los tipos de datos, la cantidad de valores nulos y los posibles registros duplicados. Durante todo este proceso se conservó `df_raw` sin modificaciones, para mantener una copia del conjunto de datos original.

### ¿Por qué se tomaron estas decisiones?

- **Fallar pronto y con un mensaje claro.** Si falta alguna columna o existen identificadores repetidos, las operaciones posteriores, como la partición, la vectorización y la generación de recomendaciones, podrían producir resultados incorrectos. Por esta razón, se prefirió detectar estos problemas al comienzo del proceso.

- **Utilizar `coerce` en lugar de interrumpir la carga.** Cuando un valor no puede convertirse a formato numérico, se transforma en un nulo que posteriormente puede contabilizarse en la auditoría. Esto permite identificar los problemas sin interrumpir la lectura del archivo ni mantener valores numéricos almacenados como texto.

- **Revisar los datos antes de definir las reglas de limpieza.** Las decisiones tomadas en los siguientes pasos se basan en las características observadas en el CSV y no en supuestos sobre cómo deberían comportarse sus variables.

### Resultados de la exploración inicial

El conjunto de datos contiene **17.978 películas y 16 columnas**. A partir de las comprobaciones realizadas se obtuvieron los siguientes resultados:

- `movie_id` presenta identificadores únicos y no se encontraron filas duplicadas.
- Se identificaron 10 valores nulos en `genres`, 494 en `production_companies` (2,75 %), 176 en `production_countries` y 92 en `spoken_languages`. Esto justifica la necesidad de considerar valores vacíos al procesar los géneros y las productoras en el paso 2.
- La variable `status` contiene únicamente registros con valor `Released`, mientras que `adult` presenta exclusivamente valores `False`. Por lo tanto, no se identificaron películas sin estrenar ni contenido adulto que requiriera filtrado.
- Las fechas de estreno abarcan desde el 01-01-2010 hasta el 30-12-2025, lo que coincide con el período indicado en el título del notebook.
- El valor mínimo de `vote_count` es 50, lo que indica que el conjunto de datos ya contempla un filtro previo de películas con muy pocos votos.
- La variable `budget` presenta 12.463 registros con valor cero, mientras que `revenue` presenta 11.156. Esto significa que una proporción importante de las películas no cuenta con información financiera registrada y justifica el tratamiento aplicado en el paso 5.
- Se encontraron 46 películas con `runtime` menor o igual a cero, 481 con duraciones entre 1 y 39 minutos y 6 con duraciones superiores a 300 minutos. Estos resultados respaldan el tratamiento específico de esta variable mediante medianas por género.

### Alternativa evaluada

Se consideró utilizar `sep=None` para permitir la detección automática del delimitador durante la lectura del archivo. Sin embargo, esta alternativa presenta un mayor costo de ejecución y puede identificar incorrectamente el separador.

Dado que en este caso la lectura estándar produjo los resultados esperados, se optó por mantenerla como método principal y utilizar la autodetección únicamente como alternativa cuando no se encuentren las columnas esperadas.


In [ ]:
def cargar_dataset(ruta):
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encontró {ruta}. Ajusta RUTA_DATOS.")
    datos = pd.read_csv(ruta)
    if set(COLUMNAS) - set(datos.columns):      # solo si falla el separador por defecto
        datos = pd.read_csv(ruta, sep=None, engine="python")
    faltan = sorted(set(COLUMNAS) - set(datos.columns))
    if faltan:
        raise ValueError(f"Faltan columnas: {faltan}")
    datos = datos.copy()
    for c in NUMERICAS:
        datos[c] = pd.to_numeric(datos[c], errors="coerce")
    if datos["movie_id"].isna().any() or datos["movie_id"].duplicated().any():
        raise ValueError("movie_id debe ser numérico, no nulo y único.")
    return datos

df_raw = cargar_dataset(RUTA_DATOS)
print(f"{len(df_raw):,} filas × {df_raw.shape[1]} columnas")
display(pd.DataFrame({"tipo":df_raw.dtypes.astype(str),
                      "nulos":df_raw.isna().sum(),
                      "% nulos":(100*df_raw.isna().mean()).round(2)}))
print("IDs únicos:", df_raw.movie_id.is_unique,
      "| filas completamente duplicadas:", df_raw.duplicated().sum())
display(df_raw.head())

## 2. Funciones auxiliares
Se aíslan las reglas de limpieza para facilitar reutilización y pruebas.


### 2. Funciones auxiliares de limpieza

#### ¿Qué se hizo?

Se implementaron funciones auxiliares para procesar los atributos textuales del conjunto de datos: `lista_generos`, `primer_genero`, `limpiar_productoras`, `limpiar_idioma` y `construir_sopa`.

Estas funciones permiten transformar los textos originales en tokens que posteriormente pueden utilizarse en la vectorización. La regla principal consiste en conservar cada nombre completo como un único token, reemplazando los espacios entre palabras por guiones bajos. Por ejemplo, `Science Fiction` se transforma en `science_fiction` y `Filmes do Equador` en `filmes_do_equador`.

En el caso de las productoras, se conservaron las primeras tres productoras de cada película, respetando sus nombres completos y evitando limitar la cantidad de palabras que componen cada nombre.

#### ¿Por qué se tomaron estas decisiones?

- **Funciones independientes y reutilizables.** Se optó por separar cada operación en funciones pequeñas para facilitar su prueba y reutilización. Estas mismas funciones se utilizan en el pipeline de la sección 4, en las pruebas individuales de la sección 9 y en las mediciones de rendimiento de la sección 10. De esta manera, se evita repetir las mismas reglas en distintos lugares y se mantiene la consistencia del procesamiento.

- **Utilización de `_` en lugar de espacios.** El `CountVectorizer` separa los textos utilizando los espacios como delimitadores. Si se conserva `science fiction`, el vectorizador interpreta ambos términos por separado, perdiendo la relación entre ellos. Al utilizar `science_fiction`, el género se mantiene como un único atributo. El mismo criterio se aplica a los nombres de películas, idiomas y productoras.

- **Conservar un máximo de tres productoras.** La mediana de productoras por película es de tres, aunque existen registros que contienen hasta 30. Considerar todas las productoras podría incorporar numerosos términos poco frecuentes, aumentando el tamaño del vocabulario y generando ruido en la representación. Por esta razón, se decidió conservar las primeras tres, manteniendo una representación más acotada de la información de producción.

- **Utilización de `unknown` para valores ausentes.** Cuando no se dispone de información sobre géneros, productoras o idioma, se incorpora un token explícito en lugar de generar una cadena vacía o interrumpir el procesamiento. Esto permite identificar los registros sin información y mantener una representación consistente.

- **Incorporación del idioma original.** La variable `original_language` se incluye como parte de la representación textual, ya que puede aportar información útil para identificar películas con características culturales o lingüísticas similares. El conjunto presenta 80 idiomas distintos y aproximadamente un 59 % de los registros corresponde al idioma inglés (`en`).

#### Corrección de un error en el tratamiento de las productoras

Durante el desarrollo se identificó y corrigió un error en la implementación inicial de `limpiar_productoras`. La versión anterior aplicaba `.split()[:3]` sobre el texto completo, lo que provocaba que se conservaran las primeras tres palabras en lugar de las primeras tres productoras.

Por ejemplo, una cadena como `LC Barreto|Filmes do Equador|...` podía terminar convertida en `lc barreto filmes`, perdiendo parte del nombre de la segunda productora y descartando las siguientes.

El problema afectaba potencialmente a los registros con múltiples productoras. En particular, 6.136 de las 17.484 películas que contienen información sobre productoras registran más de tres. La corrección consistió en separar primero las productoras utilizando el delimitador correspondiente y posteriormente seleccionar las tres primeras, procesando cada nombre completo como un único token.

Esta modificación permite conservar correctamente la identidad de las productoras y evita generar términos incompletos o genéricos que podrían perjudicar la representación de las películas.


In [ ]:
def _token(t):
    """Une las palabras de un nombre en un solo token: 'Science Fiction' -> 'science_fiction'."""
    return "_".join(str(t).strip().lower().split())

def lista_generos(x):
    if pd.isna(x) or not str(x).strip(): return ["unknown"]
    return [_token(t) for t in str(x).split("|") if t.strip()] or ["unknown"]

def limpiar_generos(x):
    return " ".join(lista_generos(x))

def primer_genero(x):
    return lista_generos(x)[0]

def limpiar_productoras(x, maximo=MAX_PRODUCTORAS):
    """Devuelve las primeras `maximo` PRODUCTORAS (no palabras), cada una como un token."""
    if pd.isna(x) or not str(x).strip(): return ""
    toks = [_token(t) for t in str(x).split("|") if t.strip()]
    return " ".join(toks[:maximo])

def limpiar_idioma(x):
    return "unknown" if pd.isna(x) or not str(x).strip() else str(x).strip().lower()

def construir_sopa(datos):
    g = datos["genres"].map(limpiar_generos)
    p = datos["production_companies"].map(limpiar_productoras)
    i = datos["original_language"].map(limpiar_idioma)
    return (g+" "+p+" "+i).str.split().str.join(" ").rename("metadata_soup")


## 3. Partición entrenamiento/prueba
Se divide antes de ajustar transformadores. Los parámetros estadísticos se aprenden exclusivamente con entrenamiento.


### 3. Partición de los datos en entrenamiento y prueba

#### ¿Qué se hizo?

Se realizó una partición aleatoria del conjunto de datos, reservando un 20 % de las películas para prueba mediante `sample(frac=0.20, random_state=42)`. Esta separación se efectuó antes de calcular cualquier parámetro estadístico o aplicar las transformaciones del pipeline.

Posteriormente, se incorporaron comprobaciones mediante `assert` para verificar que ambos conjuntos fueran independientes, que sus tamaños sumaran el total de registros originales y que no existieran identificadores de películas compartidos entre ellos.

El resultado de la partición fue el siguiente:

- **Entrenamiento:** 14.382 películas (80 %).
- **Prueba:** 3.596 películas (20 %).

#### ¿Por qué se tomó esta decisión?

- **Evitar la fuga de información (*data leakage*).** Los parámetros que se utilizan durante el procesamiento, como la media global de votos (\(C\)), el percentil de votos (\(m\)), las medianas de duración y los valores mínimos y máximos de los escaladores, deben calcularse exclusivamente con los datos de entrenamiento. Si se utilizaran todos los registros, la información del conjunto de prueba también influiría en el aprendizaje, generando una evaluación menos representativa del comportamiento del sistema frente a datos no utilizados previamente.

- **Respetar el orden del procesamiento.** La partición debe realizarse antes de ajustar el pipeline. De esta manera, los parámetros se aprenden utilizando únicamente el conjunto de entrenamiento y posteriormente se aplican al conjunto de prueba, sin volver a calcularlos.

- **Realizar distintas comprobaciones de independencia.** Se utilizaron tres verificaciones mediante `assert`: índices sin elementos compartidos, tamaños que suman el total de registros y ausencia de identificadores `movie_id` repetidos entre ambos conjuntos. Esta última comprobación es especialmente importante, ya que una misma película podría aparecer en ambos conjuntos con índices diferentes y generar una fuga de información que no sería detectada únicamente comparando los índices.

#### Decisión sobre el tipo de partición

Se optó por realizar una partición aleatoria en lugar de una separación temporal. Esta decisión responde al enfoque del proyecto, orientado a construir un recomendador basado principalmente en el contenido y los metadatos de las películas, donde no existe necesariamente una secuencia temporal que deba respetarse durante el entrenamiento.

Sin embargo, si en la Fase 4 se quisiera evaluar la capacidad del sistema para recomendar películas nuevas a partir de información histórica, sería más apropiado utilizar una partición temporal basada en `release_date`. Esto permitiría simular un escenario en el que el sistema trabaja con películas anteriores e intenta generar recomendaciones para títulos estrenados posteriormente.




In [ ]:
indices_test = df_raw.sample(frac=FRACCION_PRUEBA, random_state=SEMILLA).index
df_train = df_raw.drop(index=indices_test).copy()
df_test = df_raw.loc[indices_test].copy()
assert set(df_train.index).isdisjoint(set(df_test.index))
assert len(df_train)+len(df_test)==len(df_raw)
assert set(df_train.movie_id).isdisjoint(set(df_test.movie_id))
print("Train:", len(df_train), "| test:", len(df_test))

## 4. Arquitectura POO
`Transformador` define el contrato `ajustar()` / `transformar()`. Las subclases implementan su aprendizaje y aplicación. `Pipeline` compone transformadores diferentes mediante polimorfismo. Cada método devuelve una copia y no modifica el DataFrame original.


### 4. Arquitectura orientada a objetos: transformadores y pipeline

#### ¿Qué se hizo?

Se diseñó una estructura orientada a objetos basada en una clase abstracta `Transformador`, que establece dos operaciones principales: `ajustar()`, encargada de aprender los parámetros necesarios a partir de los datos, y `transformar()`, encargada de aplicar las transformaciones utilizando dichos parámetros.

A partir de esta estructura se implementaron clases independientes para cada regla de procesamiento y una clase `Pipeline`, que permite encadenar los distintos transformadores y ejecutar las operaciones en el orden establecido.

También se incorporó el uso de copias de los DataFrames para evitar modificaciones sobre los datos originales durante el procesamiento.

#### ¿Por qué se tomó esta decisión?

- **Separar el aprendizaje de la aplicación.** Esta separación permite evitar la fuga de información. El método `ajustar()` se ejecuta únicamente sobre los datos de entrenamiento, mientras que `transformar()` puede utilizarse tanto en entrenamiento como en prueba, empleando los parámetros previamente calculados. En la sección 6 se comprueba que la transformación no modifica los parámetros aprendidos.

- **Utilizar polimorfismo.** El `Pipeline` no necesita conocer los detalles internos de cada transformador, sino que todos deben cumplir una misma estructura de métodos. Esto permite incorporar nuevos pasos en la Fase 4, como otro escalador o una transformación adicional, sin tener que modificar la implementación general del pipeline.

- **Definir una clase abstracta.** El uso de `ABC` y `@abstractmethod` establece una estructura común para todos los transformadores. Si una clase hija no implementa los métodos abstractos requeridos, Python impide crear una instancia de ella. Esto permite detectar implementaciones incompletas antes de ejecutar el procesamiento.

- **Proteger los parámetros aprendidos.** Los parámetros se almacenan en el atributo `_parametros` y se consultan mediante la propiedad `parametros`, que devuelve una copia. De esta manera, se reduce el riesgo de modificar accidentalmente el estado interno de los transformadores desde fuera de sus clases.

- **Evitar la modificación de los datos originales.** Se mantiene `df_raw` sin alteraciones para poder recuperar el conjunto original en cualquier momento. Además, se ajustó el método `transformar()` para evitar realizar copias innecesarias, considerando que cada método `aplicar()` ya trabaja sobre su propia copia de los datos.

#### Decisiones tomadas en cada transformador

- **`ImputarRuntimePorGenero`:** la duración habitual de una película puede variar considerablemente entre géneros, por lo que utilizar una mediana específica para cada grupo resulta más representativo que aplicar un único valor global. Se eligió la mediana en lugar de la media debido a la presencia de valores extremos, con duraciones de hasta 585 minutos.

  Para calcular las medianas se consideran únicamente duraciones entre 40 y 300 minutos. Esto evita que posibles errores de registro, como valores de 2 o 7 minutos, afecten los resultados, especialmente en grupos pequeños. Además, se estableció un mínimo de 30 películas por género; cuando no se alcanza esa cantidad, se utiliza la mediana global.

  Los valores atípicos no se eliminan, sino que se identifican mediante la variable `runtime_atipico`, ya que una duración poco habitual no implica necesariamente que el registro sea incorrecto. Por ejemplo, un cortometraje puede tener una duración inferior a la de un largometraje sin que esto represente un error.

- **`CerosComoAusentes`:** en este conjunto de datos, los valores `budget = 0` y `revenue = 0` se interpretan como información no disponible y no como presupuestos o ingresos realmente nulos. También se identificaron valores muy pequeños, como 1, 12 o 500, que podrían corresponder a registros incompletos o marcadores de ausencia. En particular, existen 35 presupuestos y 45 ingresos con valores inferiores a 1.000. Por esta razón, se estableció dicho umbral para tratar estos casos como valores ausentes. Esta decisión responde a las características observadas en el dataset.

- **`ConstructorSopa`:** reúne los géneros, las productoras y el idioma original de cada película en una única cadena de texto. Esta representación permite utilizar posteriormente técnicas de vectorización como *Bag of Words*, facilitando la comparación del contenido entre películas.

- **`CalidadPonderada`:** combina la calificación promedio de cada película con la cantidad de votos recibidos mediante el método Weighted Rating. Su funcionamiento, junto con la justificación y selección de sus parámetros, se desarrolla en el paso 5.

- **`LogNaturalPlus1`:** la variable `popularity` presenta una distribución fuertemente asimétrica, con una media de 2,07, un máximo de 103,9 y una asimetría de 9,2. Se aplica la transformación `log1p` para reducir esta asimetría, obteniendo un valor de 1,56. Además, permite trabajar con valores iguales a cero. Se incorporó también una validación para impedir valores negativos, ya que no son válidos para esta transformación.

- **`EscaladorMinMax`:** transforma las puntuaciones para llevarlas al intervalo [0, 1]. Esto permite trabajar con variables que originalmente tienen escalas diferentes y facilita su posterior combinación en la Fase 4. Cuando el valor máximo y el mínimo coinciden, se devuelve 0 para evitar una división por cero.


In [ ]:
class Transformador(ABC):
    def __init__(self, nombre):
        self.nombre, self._ajustado, self._parametros = nombre, False, {}
    @property
    def parametros(self): return dict(self._parametros)
    @abstractmethod
    def aprender(self, datos): pass
    @abstractmethod
    def aplicar(self, datos): pass
    def ajustar(self, datos):
        self._parametros = self.aprender(datos)
        self._ajustado = True
        return self
    def transformar(self, datos):
        if not self._ajustado: raise RuntimeError(f"{self.nombre}: ejecutar ajustar() primero.")
        return self.aplicar(datos)   # cada aplicar() ya trabaja sobre una copia
    def ajustar_transformar(self, datos): return self.ajustar(datos).transformar(datos)


class ImputarRuntimePorGenero(Transformador):
    def __init__(self): super().__init__("ImputarRuntimePorGenero")
    def aprender(self, datos):
        t = datos[["runtime","genres"]].copy()
        t["genero"] = t.genres.map(primer_genero)
        v = t[t.runtime.between(RUNTIME_MIN, RUNTIME_MAX)]
        if v.empty: raise ValueError("No hay runtime plausible para aprender medianas.")
        g = v.groupby("genero").runtime.agg(["median","size"])
        g = g[g["size"] >= MIN_GRUPO_RUNTIME]          # grupos pequeños -> mediana global
        return {"global":float(v.runtime.median()),
                "generos":g["median"].to_dict(),
                "rango_valido":[RUNTIME_MIN, RUNTIME_MAX],
                "min_grupo":MIN_GRUPO_RUNTIME}
    def aplicar(self, datos):
        out = datos.copy()
        g = out.genres.map(primer_genero)
        mask = out.runtime.isna() | (out.runtime <= 0)
        imputada = g.map(self._parametros["generos"])
        out["runtime_clean"] = out.runtime.astype(float)   # evita TypeError/truncamiento (runtime es int64)
        out["runtime_atipico"] = ~out.runtime.between(RUNTIME_MIN, RUNTIME_MAX)
        out.loc[mask,"runtime_clean"] = imputada[mask].fillna(self._parametros["global"])
        return out


class CerosComoAusentes(Transformador):
    def __init__(self, columna, umbral=UMBRAL_DINERO):
        super().__init__(f"CerosComoAusentes({columna})"); self.columna=columna; self.umbral=umbral
    def aprender(self, datos):
        if self.columna not in datos: raise KeyError(self.columna)
        return {"columna":self.columna,"umbral":self.umbral}
    def aplicar(self, datos):
        out=datos.copy()
        # 0 y valores < umbral (marcadores tipo 1, 12, 500) se tratan como ausentes
        out[self.columna+"_clean"]=out[self.columna].mask(out[self.columna].lt(self.umbral))
        return out


class ConstructorSopa(Transformador):
    def __init__(self): super().__init__("ConstructorSopa")
    def aprender(self, datos):
        faltan={"genres","production_companies","original_language"}-set(datos.columns)
        if faltan: raise KeyError(f"Faltan columnas: {faltan}")
        return {"max_productoras":MAX_PRODUCTORAS}
    def aplicar(self, datos):
        out=datos.copy(); out["metadata_soup"]=construir_sopa(out); return out


class CalidadPonderada(Transformador):
    def __init__(self, percentil=.75):
        super().__init__("CalidadPonderada"); self.percentil=percentil
    def aprender(self, datos):
        v=datos.loc[datos.vote_average.notna() & datos.vote_count.notna() & (datos.vote_count>=0)]
        if v.empty: raise ValueError("No hay valoraciones válidas.")
        return {"C":float(v.vote_average.mean()),
                "m":float(v.vote_count.quantile(self.percentil)),
                "percentil":self.percentil}
    def aplicar(self, datos):
        out=datos.copy(); C=self._parametros["C"]; m=self._parametros["m"]
        v=out.vote_count.fillna(0).clip(lower=0); R=out.vote_average.fillna(C)
        out["weighted_rating"] = R if m<=0 else (v/(v+m))*R+(m/(v+m))*C
        return out


class LogNaturalPlus1(Transformador):
    def __init__(self, columna="popularity"):
        super().__init__(f"LogNaturalPlus1({columna})"); self.columna=columna
    def aprender(self, datos):
        if self.columna not in datos: raise KeyError(self.columna)
        if (datos[self.columna].dropna()<0).any(): raise ValueError("log1p no admite negativos.")
        return {"columna":self.columna}
    def aplicar(self, datos):
        out=datos.copy()
        if (out[self.columna].dropna()<0).any(): raise ValueError("Se detectaron negativos.")
        out[self.columna+"_log"]=np.log1p(out[self.columna])
        return out


class EscaladorMinMax(Transformador):
    def __init__(self, columna, destino):
        super().__init__(f"EscaladorMinMax({columna})")
        self.columna,self.destino=columna,destino
    def aprender(self, datos):
        s=datos[self.columna].dropna()
        if s.empty: raise ValueError(f"Sin datos válidos: {self.columna}")
        return {"min":float(s.min()),"max":float(s.max())}
    def aplicar(self, datos):
        out=datos.copy(); a=self._parametros["min"]; b=self._parametros["max"]
        out[self.destino]=0.0 if a==b else (out[self.columna]-a)/(b-a)
        return out


class Pipeline:
    def __init__(self, pasos):
        if not all(isinstance(p,Transformador) for p in pasos):
            raise TypeError("Todos los pasos deben heredar de Transformador.")
        self._pasos=list(pasos); self._ajustado=False
    def ajustar(self, datos):
        actual=datos.copy()
        for paso in self._pasos:
            paso.ajustar(actual); actual=paso.transformar(actual)
        self._ajustado=True
        return self
    def transformar(self, datos):
        if not self._ajustado: raise RuntimeError("Pipeline sin ajustar.")
        actual=datos.copy()
        for paso in self._pasos: actual=paso.transformar(actual)
        return actual
    def resumen(self):
        return pd.DataFrame([{"orden":i,"paso":p.nombre,"clase":type(p).__name__}
                             for i,p in enumerate(self._pasos,1)])
    def parametros(self): return {p.nombre:p.parametros for p in self._pasos}

## 5. Construcción y ajuste del pipeline
Se integran elementos del primer notebook (duración por género, ceros como ausentes, sopa y Weighted Rating) con la separación train/test del segundo. Se escalan calidad ponderada y popularidad logarítmica.

### 5. Construcción y ajuste del pipeline

#### ¿Qué se hizo?

Se construyó el pipeline de preparación de datos mediante la integración de ocho transformadores, organizados en el siguiente orden:

1. Imputación de valores ausentes en `runtime`.
2. Tratamiento de ceros en `budget`.
3. Tratamiento de ceros en `revenue`.
4. Construcción de la representación textual (*sopa*).
5. Cálculo de Weighted Rating.
6. Transformación logarítmica de `popularity`.
7. Escalamiento de `weighted_rating`.
8. Escalamiento de `popularity_log`.

Una vez definido el orden de las operaciones, el pipeline se ajustó exclusivamente con `df_train`. Posteriormente, se utilizó para transformar tanto el conjunto de entrenamiento como el de prueba, manteniendo los parámetros aprendidos durante el ajuste inicial.

#### ¿Por qué se eligió este orden?

- **Escaladores al final del pipeline.** Los dos últimos transformadores dependen de columnas que se generan durante los pasos anteriores: `weighted_rating` y `popularity_log`. Por esta razón, deben ejecutarse después de sus respectivas transformaciones. Alterar este orden provocaría errores al intentar acceder a columnas que todavía no existen.

- **Ajuste secuencial de los transformadores.** Cada paso del pipeline se ajusta utilizando el resultado producido por el transformador anterior, en lugar de trabajar directamente sobre el DataFrame original. Esto permite que cada transformación aprenda sus parámetros considerando los datos tal como llegan a esa etapa del procesamiento.

- **Ajuste exclusivo con entrenamiento.** El pipeline se ajusta únicamente con `df_train`. Una vez aprendidos los parámetros, estos se mantienen para transformar tanto entrenamiento como prueba, evitando que la información del conjunto de prueba intervenga en el proceso de aprendizaje.


#### ¿Por qué calcular \(C\) y \(m\) únicamente con entrenamiento?

Ambos parámetros se obtienen a partir de la distribución de los datos, por lo que deben calcularse exclusivamente con el conjunto de entrenamiento.

Utilizar también los registros de prueba para obtener la media global o el percentil de votos implicaría incorporar información que debería permanecer independiente durante la evaluación. Esto produciría una fuga de información y podría afectar la interpretación de los resultados posteriores.

Por esta razón, una vez calculados \(C\) y \(m\), sus valores se mantienen durante la transformación del conjunto de prueba, sin volver a estimarlos.


In [ ]:
def construir_pipeline():
    return Pipeline([
        ImputarRuntimePorGenero(),
        CerosComoAusentes("budget"),
        CerosComoAusentes("revenue"),
        ConstructorSopa(),
        CalidadPonderada(PERCENTIL_M),
        LogNaturalPlus1("popularity"),
        EscaladorMinMax("weighted_rating","score_quality_norm"),
        EscaladorMinMax("popularity_log","score_popularity_norm")
    ])

pipeline=construir_pipeline().ajustar(df_train)
train_t=pipeline.transformar(df_train)
test_t=pipeline.transformar(df_test)
display(pipeline.resumen())
print("Dimensiones train/test:",train_t.shape,test_t.shape)
print(json.dumps(pipeline.parametros(),indent=2,ensure_ascii=False,default=str))

### 🔍 Cómo leer la salida del paso 5

- La tabla de pasos confirma el orden de ejecución. El tamaño `(14382, 25)` y `(3596, 25)` muestra que ambos conjuntos reciben las mismas columnas nuevas (`runtime_clean`, `runtime_atipico`, `budget_clean`, `revenue_clean`, `metadata_soup`, `weighted_rating`, `popularity_log` y los dos puntajes normalizados).
- En el JSON de parámetros verás `C ≈ 6,327` y `m = 421`: son los valores que se reutilizan tal cual sobre la prueba.
- Que no aparezca el género `unknown` en las medianas es intencional: tiene muy pocas películas con duración plausible y por eso usa la mediana global.

## 6. Validación de fuga de información y conservación del estado
Los extremos de prueba pueden salir de [0,1] sin ser un error: se aplican los parámetros de entrenamiento, no se recalculan.


### 6. Validaciones de fuga de datos y estado del pipeline

#### ¿Qué se hizo?

Se implementaron aserciones para comprobar el comportamiento del pipeline y verificar que las transformaciones no alteren información que debe mantenerse estable. Las validaciones comprueban que:

1. El orden de los `movie_id` se conserve después de cada transformación.
2. Ninguna sopa de características quede vacía.
3. Ningún valor de `runtime_clean` sea nulo o menor o igual a cero.
4. Los puntajes calculados sean valores finitos.
5. En el conjunto de entrenamiento, ambos puntajes normalizados alcancen exactamente los valores mínimo y máximo de 0 y 1.
6. Volver a transformar el conjunto de prueba no modifique los parámetros aprendidos durante el entrenamiento.

#### ¿Por qué se tomaron estas decisiones?

- **Conservar el orden de las películas:** si alguna transformación reordenara las filas, se podrían alterar las asociaciones entre cada película y sus respectivos puntajes sin que necesariamente se produjera un error visible. Por eso, se verifica que los `movie_id` mantengan el mismo orden durante todo el proceso.

- **Verificar los valores mínimo y máximo en entrenamiento:** como el escalador aprende sus rangos a partir del propio conjunto de entrenamiento, los valores normalizados deben alcanzar exactamente 0 y 1 en ese conjunto. Esta comprobación es más estricta que verificar únicamente que los resultados estén dentro del intervalo [0, 1], ya que esa condición podría cumplirse incluso si existiera un error en el ajuste del escalador.

- **Comprobar que los parámetros no cambien:** se verifica que aplicar nuevamente las transformaciones al conjunto de prueba no modifique los parámetros aprendidos, como la media global (C), el umbral (m) o los rangos de escalamiento. Esto permite comprobar que el conjunto de prueba se utiliza únicamente para aplicar las transformaciones y no para aprender nuevos valores.

#### ¿Por qué es correcto que algunos valores de prueba queden fuera de [0, 1]?

El escalador aplica al conjunto de prueba los rangos calculados exclusivamente con los datos de entrenamiento. Por lo tanto, si una película de prueba presenta un puntaje superior o inferior a los valores observados durante el entrenamiento, su resultado normalizado puede quedar fuera del intervalo [0, 1].

Este comportamiento es esperado y no representa un error. Recalcular los rangos utilizando el conjunto de prueba modificaría los parámetros del modelo e introduciría fuga de datos. Por este motivo, se mantienen los rangos aprendidos durante el entrenamiento, incluso cuando algunos valores de prueba quedan fuera del intervalo establecido.


In [ ]:
params_antes=pipeline.parametros()
assert train_t.movie_id.tolist()==df_train.movie_id.tolist()
assert test_t.movie_id.tolist()==df_test.movie_id.tolist()
assert train_t.metadata_soup.str.len().gt(0).all()
assert train_t.runtime_clean.gt(0).all()
assert np.isfinite(train_t[["score_quality_norm","score_popularity_norm"]]).all().all()
for c in ["score_quality_norm","score_popularity_norm"]:
    assert train_t[c].between(0,1).all()
    assert np.isclose(train_t[c].min(),0) and np.isclose(train_t[c].max(),1)   # min/max exactos por construcción
assert not train_t.runtime_clean.isna().any()
_ = pipeline.transformar(df_test)
assert params_antes==pipeline.parametros()
display(pd.DataFrame({
    "variable":["score_quality_norm","score_popularity_norm"],
    "min_test":[test_t[c].min() for c in ["score_quality_norm","score_popularity_norm"]],
    "max_test":[test_t[c].max() for c in ["score_quality_norm","score_popularity_norm"]],
    "fuera_01":[(~test_t[c].between(0,1)).sum() for c in ["score_quality_norm","score_popularity_norm"]]
}))
print("OK: IDs conservados, train normalizado y parámetros inalterados.")

### 🔍 Cómo leer la salida del paso 6

La tabla final muestra los extremos en prueba. En `score_quality_norm` hay **3 películas** fuera de [0, 1] (mínimo −0,023; máximo 1,012) y en `score_popularity_norm` ninguna. Son desviaciones mínimas, no un error. Si en la Fase 4 se combinan puntajes con pesos, conviene decidir explícitamente si se recortan a [0, 1] (*clipping*).

## 7. Vectorización dispersa
El vocabulario se aprende solo en entrenamiento y se reutiliza para transformar prueba.


**Qué se hizo.** Se convirtió la sopa de cada película en un vector de conteos mediante `CountVectorizer`, ajustando el vocabulario exclusivamente con los datos de entrenamiento y utilizando `token_pattern=r"\S+"` (un token corresponde a cualquier secuencia de caracteres sin espacios). Como resultado, se obtuvieron matrices de 14.382 × 12.069 (train) y 3.596 × 12.069 (test), ambas en formato disperso CSR.

**Por qué.**

- *Vocabulario aprendido solo en entrenamiento.* Se mantuvo el criterio de prevención de fuga de información: el vocabulario se construyó exclusivamente a partir del conjunto de entrenamiento, mientras que los datos de prueba se representaron utilizando las mismas columnas. Los tokens presentes únicamente en prueba se ignoran, garantizando que ambas matrices tengan el mismo número de columnas, lo que se comprobó mediante un `assert`. Además, se verificó que ninguna fila del conjunto de prueba quedara representada por un vector nulo.

- *`token_pattern=r"\S+"`.* El patrón predeterminado de scikit-learn descarta tokens de un solo carácter y puede dividirlos según determinados signos de puntuación. Esto habría provocado la pérdida o fragmentación de tokens como `science_fiction` o `m6_films`. Por este motivo, se utilizó un patrón que permite conservarlos íntegramente.

- *Matriz dispersa.* Una matriz densa de 14.382 × 12.069 contendría aproximadamente 173 millones de celdas, lo que requeriría alrededor de 1,4 GB en formato `float64`. Sin embargo, la gran mayoría de sus valores son cero, dado que la densidad real es de apenas un 0,046 %. Por ello, se utilizó el formato disperso CSR, que almacena principalmente los valores distintos de cero y permite reducir considerablemente el consumo de memoria, facilitando los cálculos de similitud previstos para la Fase 4.

- *Conteo en lugar de TF-IDF (por ahora).* Se optó por utilizar conteos simples, ya que cada token aparece normalmente una sola vez por película, por lo que la representación resultante es prácticamente binaria y fácil de interpretar. Aunque TF-IDF constituye una alternativa razonable para reducir la influencia de tokens muy frecuentes, como `en` o `drama`, su incorporación representa una decisión de modelado que corresponde evaluar durante la Fase 4, evitando anticipar modificaciones sin evidencia suficiente.

**Efecto de la corrección de la sopa.** El vocabulario aumentó de 7.935 a 12.069 tokens. Este incremento se explica porque, tras corregir el procesamiento de la sopa, las productoras se conservaron completas y diferenciadas, en lugar de quedar reducidas a fragmentos genéricos.

In [ ]:
vectorizador=CountVectorizer(token_pattern=r"\S+")   # conserva tokens como science_fiction y m6_films
matriz_train=vectorizador.fit_transform(train_t.metadata_soup)
matriz_test=vectorizador.transform(test_t.metadata_soup)
assert matriz_train.shape[0]==len(train_t) and matriz_test.shape[0]==len(test_t)
assert matriz_train.shape[1]==matriz_test.shape[1]
print("Train:",matriz_train.shape,matriz_train.getformat(),
      "| Test:",matriz_test.shape,matriz_test.getformat())
print("Densidad train:",round(matriz_train.nnz/np.prod(matriz_train.shape),6))

## 8. Recursividad aplicada a metadatos jerárquicos
El CSV actual usa cadenas separadas por `|`, no listas anidadas. Por eso la recursividad se demuestra sobre una estructura sintética representativa de posibles metadatos JSON futuros. Ambas soluciones deben preservar el orden y omitir `None`.


**Qué se hizo.** Se implementó el aplanado de una estructura anidada mediante dos enfoques: uno recursivo y otro iterativo con una pila explícita. Posteriormente, se comprobó que ambas implementaciones producen el mismo resultado, preservan el orden original de los elementos y omiten los valores `None`.

**Por qué.**

- *Contenido del CSV.* Los metadatos disponibles en el dataset corresponden a cadenas planas separadas por `|`, por lo que no contienen estructuras de listas anidadas reales. En consecuencia, el comportamiento de la recursividad se demostró utilizando una estructura **sintética**, dejando explícitamente establecido en el notebook que se trata de un ejercicio demostrativo y no de una transformación necesaria para los datos actuales.

- *Aplicación futura.* Aunque la estructura actual del CSV no requiere este procedimiento, el aplanado recursivo podría resultar útil si en futuras versiones los metadatos se obtuvieran directamente desde una API y estuvieran organizados en estructuras jerárquicas, como objetos JSON con listas anidadas.

- *Comparación de dos implementaciones.* La versión recursiva representa de manera natural la jerarquía de la estructura, pero está limitada por la profundidad máxima de recursión de Python. Por otro lado, la versión iterativa utiliza una pila explícita para controlar el recorrido. Implementar ambas y comprobar su equivalencia permite comprender sus diferencias y seleccionar el enfoque más adecuado según las características de los datos.

- *Control mediante `max_profundidad`.* Se incorporó un parámetro de profundidad máxima para detectar estructuras excesivamente anidadas y generar un `RecursionError` explícito y controlado, en lugar de permitir que se produzca un desbordamiento de recursión inesperado. Este comportamiento también se verifica mediante las pruebas de la sección 9.

- *Uso de `reversed` en la versión iterativa.* Debido a que una pila sigue el principio LIFO (*Last In, First Out*), sus elementos se extraen en orden inverso al de inserción. Por este motivo, se utiliza `reversed` al incorporar los elementos a la pila, de manera que el recorrido final conserve el orden original de la estructura.

In [ ]:
def aplanar_recursivo(estructura, profundidad=0, max_profundidad=500):
    if profundidad>max_profundidad: raise RecursionError("Profundidad máxima excedida.")
    resultado=[]
    if isinstance(estructura,(list,tuple)):
        for elemento in estructura:
            resultado.extend(aplanar_recursivo(elemento,profundidad+1,max_profundidad))
    elif estructura is not None:
        resultado.append(estructura)
    return resultado

def aplanar_iterativo(estructura):
    resultado=[]; pila=[estructura]
    while pila:
        actual=pila.pop()
        if isinstance(actual,(list,tuple)): pila.extend(reversed(actual))
        elif actual is not None: resultado.append(actual)
    return resultado

caso=["Drama",["Thriller",["Romance"]],None,("Comedy",),["Action"]]
rec=aplanar_recursivo(caso); ite=aplanar_iterativo(caso)
assert rec==ite
print("Recursivo:",rec,"| Iterativo:",ite,"| equivalencia: OK")

### 8.1 Complejidad teórica

Sea \(N\) el total de elementos visitados, \(h\) la profundidad máxima y \(p\) el máximo de elementos pendientes en la pila iterativa.

- Tiempo de ambas versiones: \(O(N)\), cada elemento se visita una vez.
- Espacio recursivo auxiliar: \(O(h)\) para la pila de llamadas, además de \(O(N)\) para el resultado.
- Espacio iterativo auxiliar: hasta \(O(p)\) para la pila explícita, además de \(O(N)\) para el resultado.

La versión recursiva refleja naturalmente la jerarquía, pero está limitada por la profundidad de recursión de Python. La iterativa evita ese límite práctico mediante una pila explícita.

## 9. Pruebas funcionales, casos límite y excepciones


**Qué se hizo.** Se implementó la función auxiliar `debe_fallar(tipo, función)`, cuya finalidad es exigir que una operación genere exactamente la excepción esperada. Esta herramienta se utilizó para comprobar distintos errores previstos: ejecución de un pipeline sin ajustar, ausencia de columnas requeridas, incorporación de pasos que no corresponden a un `Transformador`, presencia de valores negativos al aplicar `log1p` y exceso de profundidad en la recursión. Además, se incorporaron pruebas específicas para verificar las correcciones realizadas durante el desarrollo.

**Por qué.**

- *Comportamiento ante datos incorrectos.* Un programa correctamente implementado no solo debe funcionar cuando recibe datos válidos, sino también responder de manera predecible ante entradas incorrectas o situaciones no contempladas. Comprobar la excepción exacta permite evitar que un error diferente al esperado sea interpretado como un resultado satisfactorio.

- *Funcionamiento de `debe_fallar`.* La función auxiliar distingue tres posibles resultados: la excepción esperada, que se considera una prueba satisfactoria (OK); una excepción diferente, que se informa indicando su tipo y mensaje; y la ausencia de cualquier excepción, que se registra como un fallo. De esta manera, las pruebas no pueden aprobar simplemente porque la operación haya generado algún tipo de error.

- *Comprobación de casos límite.* Se incorporaron situaciones particulares para verificar el comportamiento de las transformaciones ante entradas especiales, como listas vacías, valores `None` dentro de estructuras anidadas, géneros nulos y productoras nulas. Estos casos permiten detectar comportamientos inesperados que podrían pasar inadvertidos al trabajar únicamente con datos habituales.

- *Pruebas de regresión de las correcciones.* Se añadieron pruebas explícitas para los errores identificados y corregidos durante el desarrollo: construcción de la sopa con nombres de productoras de varias palabras, conservación de géneros compuestos, imputación cuando no existe el grupo `unknown` y aplicación del umbral mínimo de dinero. Estas pruebas permiten verificar que las correcciones se mantengan en futuras modificaciones y que, si alguno de los defectos vuelve a introducirse, el notebook lo detecte inmediatamente en lugar de generar resultados incorrectos sin advertencia.

In [ ]:
def debe_fallar(tipo, funcion):
    try: funcion()
    except tipo: return "OK"
    except Exception as e: raise AssertionError(f"Se esperaba {tipo.__name__}, ocurrió {type(e).__name__}: {e}")
    raise AssertionError(f"Se esperaba {tipo.__name__} y no hubo excepción.")

assert aplanar_recursivo([])==[] and aplanar_iterativo([])==[]
assert aplanar_recursivo([None,["Drama"]])==["Drama"]
debe_fallar(RuntimeError,lambda: construir_pipeline().transformar(df_train))
debe_fallar(KeyError,lambda: ConstructorSopa().ajustar(df_train.drop(columns=["genres"])))
debe_fallar(TypeError,lambda: Pipeline(["no es transformador"]))
debe_fallar(ValueError,lambda: LogNaturalPlus1().ajustar(pd.DataFrame({"popularity":[-1,2]})))
debe_fallar(RecursionError,lambda: aplanar_recursivo([[[["x"]]]],max_profundidad=1))
# --- pruebas de las correcciones ---
_f=pd.DataFrame({"genres":["Science Fiction|TV Movie"],
                 "production_companies":["LC Barreto|Filmes do Equador|InterVideo Productions Limited|Globo Filmes"],
                 "original_language":["pt"]})
assert construir_sopa(_f).iloc[0]=="science_fiction tv_movie lc_barreto filmes_do_equador intervideo_productions_limited pt"
assert construir_sopa(pd.DataFrame({"genres":[None],"production_companies":[None],"original_language":[None]})).iloc[0]=="unknown unknown"
_s=ImputarRuntimePorGenero().ajustar(df_train).transformar(df_train)
assert _s.runtime_clean.gt(0).all() and "unknown" not in ImputarRuntimePorGenero().ajustar(df_train).parametros["generos"]
assert CerosComoAusentes("budget").ajustar(df_train).transformar(df_train).budget_clean.dropna().ge(UMBRAL_DINERO).all()
print("Todas las pruebas funcionales y de excepciones finalizaron correctamente.")

## 10. Eficiencia: `iterrows` frente a operaciones por columnas

Antes de medir se exige equivalencia de resultados. Se informa mediana de varias ejecuciones. La memoria se mide por separado porque `tracemalloc` altera el tiempo.

Si \(N\) es el número de películas y \(L\) la longitud media de metadatos, ambas estrategias procesan aproximadamente \(O(N\cdot L)\) información. Diferencias de tiempo real no implican distinto orden asintótico.


**Qué se hizo.** Se construyó la sopa de cada película mediante dos estrategias: un recorrido fila por fila utilizando `iterrows` y una implementación basada en operaciones sobre columnas completas. Posteriormente, se compararon los tiempos de ejecución y el consumo de memoria de ambas alternativas.

**Por qué.**

- *Equivalencia antes de la medición.* Se utilizó `assert_series_equal` para comprobar que ambas implementaciones produjeran exactamente la misma serie. Esta verificación es fundamental, ya que comparar el rendimiento de dos procedimientos que generan resultados diferentes carecería de sentido.

- *Mediana de varias ejecuciones.* Se utilizó la mediana de cinco repeticiones para medir el tiempo de ejecución. Un único registro puede verse afectado por fluctuaciones del sistema y otros procesos en segundo plano, mientras que la mediana permite obtener una estimación más robusta frente a estos valores atípicos.

- *Medición de memoria independiente.* Se evaluó el consumo de memoria mediante `tracemalloc` en una medición separada del tiempo de ejecución, debido a que esta herramienta introduce sobrecarga y puede ralentizar el procesamiento. Separar ambas mediciones evita distorsionar los resultados del benchmark.

- *Mismos auxiliares en ambas implementaciones.* Se utilizaron las mismas funciones auxiliares de limpieza que forman parte del pipeline, garantizando que ambas estrategias aplicaran idénticas reglas de procesamiento. De esta manera, la comparación se concentra exclusivamente en la estrategia de recorrido y no en diferencias de implementación o limpieza.

- *Cautela en la interpretación de los resultados.* Ambas estrategias procesan una cantidad de información del orden de \(O(N \cdot L)\), donde \(N\) representa el número de películas y \(L\) la longitud media de sus metadatos. Por lo tanto, ninguna modifica el orden de complejidad asintótica; las diferencias observadas corresponden principalmente a las constantes asociadas a cada implementación. En la ejecución realizada, una de las estrategias resultó aproximadamente seis veces más rápida que la otra. Sin embargo, esta diferencia constituye un resultado empírico condicionado por el entorno de ejecución y la implementación utilizada, y no una garantía teórica de rendimiento. Esta distinción queda explícitamente señalada en el notebook.

In [ ]:
def sopa_iterrows(datos):
    salida=[]
    for _,fila in datos.iterrows():
        g=limpiar_generos(fila["genres"])
        p=limpiar_productoras(fila["production_companies"])
        i=limpiar_idioma(fila["original_language"])
        salida.append(" ".join(f"{g} {p} {i}".split()))
    return pd.Series(salida,index=datos.index,name="metadata_soup")

def sopa_columnas(datos): return construir_sopa(datos)

def medir_tiempo(funcion,datos,repeticiones=5):
    tiempos=[]; resultado=None
    for _ in range(repeticiones):
        t=time.perf_counter(); resultado=funcion(datos); tiempos.append(time.perf_counter()-t)
    return resultado,statistics.median(tiempos)

def medir_memoria(funcion,datos):
    tracemalloc.start()
    try:
        resultado=funcion(datos); _,pico=tracemalloc.get_traced_memory()
    finally: tracemalloc.stop()
    return resultado,pico/(1024**2)

a=sopa_iterrows(df_train); b=sopa_columnas(df_train)
pd.testing.assert_series_equal(a,b,check_dtype=False)   # pandas 3: string vs object
_,t_filas=medir_tiempo(sopa_iterrows,df_train)
_,t_cols=medir_tiempo(sopa_columnas,df_train)
_,m_filas=medir_memoria(sopa_iterrows,df_train)
_,m_cols=medir_memoria(sopa_columnas,df_train)
benchmark_sopa=pd.DataFrame([
    {"metodo":"iterrows","mediana_s":t_filas,"memoria_pico_MiB":m_filas},
    {"metodo":"por columnas","mediana_s":t_cols,"memoria_pico_MiB":m_cols}
])
display(benchmark_sopa.round(4))
print(f"Relación de tiempos iterrows/columnas: {t_filas/t_cols:.2f}x")
print("Resultados dependientes del entorno; no constituyen prueba asintótica.")

### 🔍 Cómo leer la salida del paso 10

Es normal que la versión por columnas use **más memoria pico** que `iterrows`: construye estructuras intermedias completas (una serie por cada paso) mientras que `iterrows` procesa una fila a la vez. Es un intercambio clásico de **tiempo contra memoria**; con 17.978 filas la diferencia (unos MiB) es irrelevante, y la ganancia de tiempo es la que importa para repetir el proceso muchas veces.

## 11. Escalabilidad recursiva e iterativa
Se comparan tamaños crecientes, siete repeticiones por tamaño y memoria rastreada por separado.


**Qué se hizo.** Se evaluó el rendimiento de ambas implementaciones del aplanado (recursiva e iterativa) utilizando tamaños crecientes de entrada: 10, 50, 100, 250 y 500 repeticiones del bloque base. Para cada tamaño se realizaron siete ejecuciones y se comprobó que ambas implementaciones produjeran resultados equivalentes.

**Por qué.**

- *Evaluación del comportamiento al escalar.* Una única medición permite observar el rendimiento para un tamaño específico, pero no permite determinar cómo evoluciona al aumentar la cantidad de datos. Por este motivo, se utilizaron tamaños crecientes para analizar el comportamiento del tiempo de ejecución y observar si su crecimiento se mantiene aproximadamente proporcional al tamaño de entrada, como se espera de un algoritmo de complejidad lineal \(O(N)\), o si presenta incrementos más pronunciados.

- *Equivalencia en cada tamaño.* Se verificó que ambas implementaciones produjeran resultados idénticos para cada uno de los tamaños evaluados, y no solamente para el caso más pequeño. Esto garantiza que la comparación de rendimiento se realice entre procedimientos funcionalmente equivalentes durante todo el experimento.

- *Medición independiente de memoria.* El consumo de memoria se evaluó por separado del tiempo de ejecución, siguiendo el mismo criterio utilizado en el paso 10. Esto se debe a que las herramientas de medición de memoria introducen una sobrecarga que puede alterar los tiempos registrados.

- *Interpretación de las diferencias.* Las diferencias de rendimiento observadas entre ambas implementaciones fueron relativamente pequeñas, lo que se explica por la baja profundidad de la estructura sintética utilizada. En estas condiciones, la ventaja práctica de la implementación iterativa no necesariamente se manifiesta al aumentar la cantidad de elementos, sino principalmente cuando se trabaja con estructuras de profundidad extrema, donde la versión recursiva puede alcanzar el límite de recursión de Python. Por lo tanto, la principal ventaja de la alternativa iterativa radica en su mayor control sobre la profundidad del recorrido, más que en una mejora significativa de velocidad para los tamaños evaluados.

In [ ]:
def estructura_sintetica(n):
    bloque=[["drama",["thriller"]],["comedia",["romance"]],["action"]]
    return bloque*n

filas=[]
for n in [10,50,100,250,500]:
    entrada=estructura_sintetica(n)
    r,tr=medir_tiempo(aplanar_recursivo,entrada,7)
    i,ti=medir_tiempo(aplanar_iterativo,entrada,7)
    assert r==i
    _,mr=medir_memoria(aplanar_recursivo,entrada)
    _,mi=medir_memoria(aplanar_iterativo,entrada)
    filas.extend([
        {"elementos":len(r),"metodo":"recursivo","mediana_s":tr,"memoria_MiB":mr},
        {"elementos":len(i),"metodo":"iterativo","mediana_s":ti,"memoria_MiB":mi}
    ])
benchmark_recursividad=pd.DataFrame(filas)
display(benchmark_recursividad.round(6))

## 12. Persistencia y relectura

Se guardan los conjuntos transformados, benchmarks y configuración en `F3/results_mejorado/`, sin sobrescribir los artefactos anteriores. Se comprueban dimensiones, columnas, orden de IDs y valores numéricos seleccionados.


**Qué se hizo.** Se guardaron en `F3/results_mejorado/` los conjuntos de datos transformados, los resultados de los benchmarks y un archivo JSON con la semilla, las constantes, los parámetros aprendidos, la versión de Python y las limitaciones del trabajo. Posteriormente, se realizó una **relectura de los archivos CSV** para comprobar que conservaran la forma, las columnas, el orden de los identificadores y los valores numéricos esperados.

**Por qué.**
- *Continuidad para la Fase 4.* Los archivos persistidos permiten retomar el trabajo sin repetir todo el proceso de limpieza y transformación. Además, el JSON documenta los parámetros y las condiciones utilizadas para generar los resultados.
- *Verificación de la persistencia.* Guardar un archivo no garantiza que su contenido se haya conservado correctamente. La relectura permite detectar posibles problemas relacionados con separadores, conversiones de tipos o redondeos antes de utilizar los datos en etapas posteriores.
- *Tolerancia numérica.* Los valores de punto flotante se compararon mediante `assert_allclose` en lugar de exigir igualdad exacta, ya que la conversión de estos valores a texto y su posterior lectura puede introducir pequeñas diferencias en los últimos decimales.
- *Separación de resultados.* Se utilizó una carpeta independiente (`results_mejorado`) para mantener estos artefactos separados de los resultados anteriores y evitar sobrescrituras accidentales.
- *Documentación de limitaciones.* El JSON registra las limitaciones relevantes del trabajo, como el hecho de que `tracemalloc` no representa la totalidad de la memoria utilizada por el proceso. Esto permite delimitar el alcance de las mediciones y evitar conclusiones que excedan lo efectivamente evaluado.

In [ ]:
DIR_SALIDA.mkdir(parents=True,exist_ok=True)
cols=["movie_id","title","metadata_soup","weighted_rating","score_quality_norm",
      "score_popularity_norm","runtime_clean","runtime_atipico","budget_clean","revenue_clean"]
train_out=train_t[cols].copy(); test_out=test_t[cols].copy()
r_train=DIR_SALIDA/"train_transformado_mejorado.csv"
r_test=DIR_SALIDA/"test_transformado_mejorado.csv"
r_bench=DIR_SALIDA/"benchmarks_mejorados.csv"
r_json=DIR_SALIDA/"configuracion_mejorada.json"
train_out.to_csv(r_train,index=False); test_out.to_csv(r_test,index=False)
pd.concat([benchmark_sopa.assign(experimento="sopa"),
           benchmark_recursividad.assign(experimento="recursividad")],
          ignore_index=True,sort=False).to_csv(r_bench,index=False)
config={"semilla":SEMILLA,"fraccion_prueba":FRACCION_PRUEBA,
        "percentil_m":PERCENTIL_M,"filas_train":len(train_t),"filas_test":len(test_t),
        "parametros":pipeline.parametros(),"python":platform.python_version(),
        "limitaciones":["Recursividad probada con estructuras sintéticas.",
                        "Benchmarks dependen del entorno.",
                        "tracemalloc no mide toda la memoria nativa.",
                        "Equivalencia total con F2 requiere comparar sus archivos de referencia.",
                        "Test puede salir de [0,1] en las escalas (se aplican parámetros de entrenamiento).",
                        "runtime fuera de [40,300] no se imputa: se marca en runtime_atipico."]}
r_json.write_text(json.dumps(config,ensure_ascii=False,indent=2,default=str),encoding="utf-8")

for ruta,original in [(r_train,train_out),(r_test,test_out)]:
    leido=pd.read_csv(ruta)
    assert leido.shape==original.shape and leido.columns.tolist()==original.columns.tolist()
    assert leido.movie_id.tolist()==original.movie_id.tolist()
    np.testing.assert_allclose(leido[["weighted_rating","score_quality_norm","score_popularity_norm"]],
                               original[["weighted_rating","score_quality_norm","score_popularity_norm"]],
                               rtol=1e-9,atol=1e-9)
    print(ruta.name,leido.shape,"relectura OK")
print("Resultados guardados en:",DIR_SALIDA)

## 13. Conclusiones y alcance para Fase 4

- Se integró un pipeline reutilizable con transformaciones especializadas y parámetros encapsulados.
- \(C\), \(m\), medianas y escalas se aprenden solo con entrenamiento y se reutilizan en prueba.
- La sopa se vectoriza con una matriz dispersa cuyo vocabulario se aprende en entrenamiento.
- La recursividad se demuestra sobre una estructura jerárquica sintética; el CSV actual contiene cadenas planas separadas por `|`.
- Las mediciones son observaciones del entorno, no demostraciones de complejidad asintótica.
- Los archivos CSV y JSON exportados permiten continuar el desarrollo.

**Alcance:** este trabajo fortalece el núcleo algorítmico de preparación. La similitud entre películas, combinación final de puntuaciones, generación de recomendaciones y evaluación corresponden al desarrollo posterior de Fase 4.

## 📒 Bitácora resumida de decisiones

| Paso | Decisión | Motivo principal | Evidencia en los datos |
|---|---|---|---|
| 0 | Constantes y semilla fijas | Reproducibilidad y trazabilidad | — |
| 1 | Validar esquema e IDs; `coerce` | Fallar pronto y contabilizar | 17.978 filas, IDs únicos |
| 2 | 1 nombre = 1 token (`_`); 3 productoras | Conservar conceptos; limitar ruido | 6.136 películas con > 3 productoras |
| 3 | Partición antes de aprender | Evitar fuga de información | 14.382 train / 3.596 test |
| 4 | Clases con `ajustar`/`transformar` | Separar aprender de aplicar | — |
| 5 | Ceros y valores < 1000 como ausentes | `0` significa "sin dato" | 12.463 `budget` y 11.156 `revenue` en cero |
| 5 | Weighted Rating (C = 6,327; m = 421) | Premiar evidencia, no solo promedio | 50 votos/9,0 → 6,61; 20.000/8,0 → 7,97 |
| 5 | `log1p` en popularidad | Reducir asimetría | asimetría 9,2 → 1,56 |
| 5 | Medianas de duración por género | Duración típica depende del género | 46 con runtime ≤ 0 |
| 6 | Pruebas de fuga y de estado | Garantizar que no hay contaminación | 3 películas de test fuera de [0,1] |
| 7 | Vocabulario solo de train; matriz dispersa | Sin fuga; memoria viable | 12.069 tokens; densidad 0,046 % |
| 8 | Recursión sobre estructura sintética | El CSV no tiene anidación | — |
| 9 | Pruebas de excepciones y regresión | Fallar de forma predecible | — |
| 10–11 | Equivalencia antes de medir; mediana | Comparar justamente | iterrows ≈ 6× más lento (entorno) |
| 12 | Guardar y releer | Detectar corrupción de archivos | CSV relee sin pérdida |